# Лабораторная работа № 3. Генетический алгоритм символьной регрессии для системы Лотки–Вольтерры

**Цель.** Проанализировать генерацию (семплирование) и отбор графов структурной причинно-следственной модели:
выбрать алгоритм генерации, использовать критерий прунинга из ЛР № 2, сохранять выборку сгенерированных графов
и по ней выбрать оптимальный граф.

| Шаг из задания | Где в ноутбуке |
|---|---|
| 1. Уравнение из ЛР № 1 | §1 – система Лотки–Вольтерры, данные Hudson Bay 1900–1920 |
| 2. Критерий удаления/сохранения элемента из ЛР № 2 | §3 – прунинг по значимости вклада вершины |
| 3–4. Алгоритм генерации, описание и переработка для графа | §2 – генетический алгоритм символьной регрессии |
| 5. Запись алгоритма и обоснование корректности | §2.3 |
| 6. Целевая функция: точность, сложность, устойчивость к интервенциям | §4 |
| 7. Последовательность генерируемых графов | §7 |
| 8. Выборка и её аппроксимация выбранным графом | §9 |
| Интервенции (структурная и в данные) | §11 |

Код лежит в пакете `lab3/` и не зависит от автора: данные, дерево, подгонка, критерий прунинга,
целевая функция и ГА – отдельные модули.

In [ ]:
import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
%matplotlib inline

from lab3.data import load_hudson_bay, make_scenarios, zero_lynx, noise, noise_cond, shuffle
from lab3.expr import true_lv_model, random_model, Model, Node
from lab3.fit import IntegralProblem, fit_model, fit_trajectory
from lab3.ga import GAConfig, GeneticSR
from lab3.objective import Weights, score, objective_terms
from lab3.interventions import StabilityEvaluator, intervention_table, do_love
from lab3.prune import prune_model
from lab3.analysis import (class_representatives, canonical_lv, refine_top, chain_trajectories, trajectory_info, expanded_str, matches_lv,
                           lv_coefficients)
from lab3 import viz

RUN_GA = True            # False – загрузить сохранённый результат results/ga_result.pkl
SEED, POP, NGEN = 0, 120, 60
os.makedirs("results/figures", exist_ok=True)

## 1. Данные и модель

Система Лотки–Вольтерры (заяц $x$, рысь $y$):
$$\dot x = \alpha x - \beta xy,\qquad \dot y = \delta xy - \gamma y.$$
Данные – численность (тыс. шкур) Hudson Bay, 21 наблюдение (1900–1920), как в ЛР № 1–2.
Для эталона параметры подгонялись по траектории ОДУ (RSS ≈ 595).

In [ ]:
ds = load_hudson_bay()
true_m = true_lv_model()
true_info = trajectory_info(true_m, ds)
print("Эталон после подгонки по траектории:", true_info["model"])
print("RSS =", round(true_info["rss"], 1))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ds.t, ds.x, "o-", label="заяц x"); ax[0].plot(ds.t, ds.y, "s-", label="рысь y"); ax[0].legend(); ax[0].set_xlabel("год")
ax[0].set_title("Данные Hudson Bay")
ax[1].plot(ds.x, ds.y, "o-"); ax[1].set_xlabel("x"); ax[1].set_ylabel("y"); ax[1].set_title("Фазовый портрет данных")
plt.show()

## 2. Алгоритм генерации графов: генетический алгоритм символьной регрессии

### 2.1. Пространство моделей (априорное распределение)
Модель – **пара деревьев** $(f_1, f_2)$: $\dot x=f_1(x,y;\theta)$, $\dot y=f_2(x,y;\theta)$.
Вершины деревьев: операции из словаря $\{+,-,\times,\div\}$ (латентные вершины), переменные $x,y$
(наблюдаемые вершины) и константы-параметры $\theta_k$. Дерево превращается в **DAG** объединением одинаковых
подвыражений без параметров (например, $x\cdot y$ используется несколькими потребителями), а DAG – в бинарную
**матрицу инцидентности** $A$ (в §12 – рисунок).

Априорное распределение – **равномерное на множестве допустимых деревьев**: допустимы деревья
с глубиной ≤ 4 и не более 13 вершин в каждом уравнении, у которых суперпозиция вычислима на данных
(нет деления на ноль, inf/nan, «взрыва» значений; область значений вложенной функции лежит в области определения
внешней). Недопустимые деревья отбрасываются при порождении.

In [ ]:
rng = np.random.default_rng(3)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a in ax:
    while True:
        m = random_model(rng, 3).simplified()
        if m.size() >= 6: break
    viz.draw_dag(a, m, str(m)[:70])
fig.suptitle("Примеры случайно порождённых графов (синие – наблюдаемые, оранжевые – параметры, зелёные – операции)")
plt.show()

### 2.2. Один шаг алгоритма

Пусть $G^{*}_{g}$ – лучшая запрунённая модель поколения $g$.

1. **Оценка.** Для каждой модели настраиваются параметры (§4: интегральное согласование), считаются точность,
   сложность и устойчивость к интервенциям → $J$.
2. **Прунинг** лучших моделей критерием из ЛР № 2 (§3). Запрунённая модель возвращается в популяцию
   (элитизм) и записывается в **цепочку графов**.
3. **Порождение следующей выборки:**
   * (а) турнирный отбор → кроссовер (обмен поддеревьями) → мутация (точечная / замена поддерева / сжатие / расширение / подъём поддерева);
   * (б) **«старое запрунённое дерево» × «новое случайное»**: кроссовер $G^*_g$ со случайно порождённой моделью + мутация
     (так результат прунинга используется на следующем шаге семплирования).
4. Все оценённые структуры попадают в **архив** (выборка графов). После последнего поколения оптимальный
   граф выбирается из архива: top-10 по $J$ уточняются подгонкой по траектории ОДУ, и выбирается минимум итоговой $J$.

### 2.3. Обоснование корректности
* **Замкнутость операторов.** Кроссовер и мутации заменяют поддерево поддеревом из того же словаря, поэтому результат
  остаётся деревом над словарём; нарушения ограничений (размер, глубина, область определения) проверяются
  после каждого оператора, некорректные потомки отбрасываются.
* **Достижимость.** Любое допустимое дерево достижимо из любого другого конечной цепочкой мутаций «замена поддерева»
  (вероятность каждой > 0) – поиск не застревает в подпространстве.
* **Сохранение значения при упрощении.** Свёртка констант, нейтральные элементы и канонический порядок множителей
  не меняют вычисляемую функцию (проверено численно на 270 случайных деревьях) и дают каноническую запись,
  по которой структуры дедуплицируются в архиве.
* **Монотонность элитизма.** Лучшая модель никогда не теряется (элита копируется в следующее поколение),
  поэтому лучшая $J$ в популяции не возрастает.
* **Прунинг не ухудшает точность значимо.** Принимаются только удаления, не показавшие статистически
  значимого ухудшения (§3), и всегда уменьшающие сложность.

## 3. Критерий прунинга (из ЛР № 2)

Правило, на котором настаивал руководитель: **вершина выбрасывается, если её удаление ничего не меняет**
(базовое правило выбора моделей машинного обучения).

Для каждой вершины $n$ дерева рассматриваются кандидаты: $n\to 0$ и $n\to$ один из его детей.
Кандидат принимается, если
1. сложность строго уменьшилась;
2. **парный $t$-критерий** по квадратам невязок на интервалах $d_i=e^{2}_{i,\text{кандидат}}-e^{2}_{i,\text{полная}}$
   не показывает значимого ухудшения: $t=\bar d/(s_d/\sqrt n) < 2$, и относительный рост ошибки ≤ 10 %.

Из принятых выбирается удаление, дающее наибольшее упрощение; процедура повторяется. Коэффициент вида $c\cdot T\to T$
не считается удалением (иначе «схлопываются» осмысленные параметры).

**Демонстрация:** к модели Лотки–Вольтерры добавлен лишний член $0.01\,y\cdot y$.

In [ ]:
prob = IntegralProblem(ds)
X, Y, C = (lambda: Node.var("x")), (lambda: Node.var("y")), Node.const
f1 = Node.binop("+", Node.binop("-", Node.binop("*", C(.5), X()), Node.binop("*", C(.02), Node.binop("*", X(), Y()))),
                Node.binop("*", C(.01), Node.binop("*", Y(), Y())))
bloated = Model([f1, true_lv_model().trees[1]])
fit_model(bloated, prob)
pruned, log = prune_model(bloated, prob)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
viz.draw_dag(ax[0], bloated, "до прунинга (лишний член 0.01·y·y)")
viz.draw_dag(ax[1], pruned, "после прунинга")
plt.show()
print("принятые удаления:", log)

## 4. Целевая функция выбора графов

$$J(M)=w_{acc}\ln(\text{acc})+w_{cmp}\,C+w_{stab}\,S\ \to\ \min,$$

* **Точность** $\text{acc}$ – средняя по двум уравнениям доля необъяснённой дисперсии приращений
  $\mathrm{SSE}/\mathrm{SST}$ (аналог $1-R^2$). Параметры подгоняются **интегральным согласованием**:
  $x_{i+1}-x_i\approx\int_{t_i}^{t_{i+1}} f_1(\hat x(t),\hat y(t);\theta)\,dt$, где $\hat x,\hat y$ – интерполяция данных (PCHIP).
  Это быстро (ОДУ не решается), не требует дифференцирования шумных данных и позволяет подгонять деревья независимо.
* **Сложность** $C$ = число вершин деревьев + $0.5$ на параметр. Вес $w_{cmp}=0.03$ на вершину сопоставим с BIC-штрафом для ~40 невязок.
* **Устойчивость** $S=\ln(1+\Delta/\text{acc})$, где $\Delta$ – средний рост ошибки на **исходных** данных после переподгонки модели
  на данных с интервенциями (шум, шум, зависящий от рыси, малая выборка, смещение – те же, что в ЛР № 2, 8 возмущённых наборов).
  Тогда $\ln(\text{acc})+S=\ln(\text{acc}+\Delta)$ – логарифм ошибки на чистых данных, усреднённой по интервенциям.

Пример для эталонной модели:

In [ ]:
w = Weights()
scen = make_scenarios(ds)
stab = StabilityEvaluator(scen)
true_sc = score(true_m.copy(), prob, stab, w, restarts=2, rng=np.random.default_rng(0))
print(true_sc.summary())
print({k: round(float(v), 3) for k, v in objective_terms(true_sc, w).items()})
print("Возмущённые наборы:", [d.name for d in scen])

## 5. Запуск генетического алгоритма

In [ ]:
cfg = GAConfig(pop_size=POP, n_gen=NGEN, seed=SEED, weights=w)
pk = "results/ga_result.pkl"
if RUN_GA or not os.path.exists(pk):
    ga = GeneticSR(ds, scen, cfg)
    res = ga.run()
    pickle.dump(res, open(pk, "wb"))
else:
    res = pickle.load(open(pk, "rb"))
print(f"\nвремя {res.runtime:.0f} c; оценено моделей: {res.n_evaluated}; уникальных структур в архиве: {len(res.archive)}")

## 6. Сходимость
Лучшая и медианная $J$, разнообразие популяции, накопление выборки графов.

In [ ]:
viz.fig_convergence(res); plt.show()

## 7. Последовательность генерируемых графов

**(а) Порождаемые графы.** Для каждого поколения берётся лучший *новый* (ранее не встречавшийся) граф – это и есть
последовательность семплирования. У каждого графа: число параметров, латентных и наблюдаемых вершин, ошибка
(интегральная $1-R^2$ и RSS по траектории ОДУ), раскрытая формула и **фазовый портрет** (серые точки – данные, линия –
модель; красная – раскрытие совпадает с Лоткой–Вольтеррой). Если модель «сломана», это видно по фазовому портрету
(отрицательные значения, спирали, расходимость).

In [ ]:
traj_s = chain_trajectories(res.sampled_chain, ds, max_nfev=15)
viz.fig_chain_metrics(res.sampled_chain, traj_s, title="Лучший ПОРОЖДЁННЫЙ граф каждого поколения"); plt.show()
pages = viz.fig_chain_pages(res.sampled_chain, traj_s, ds, per_page=12, ncols=4, dedupe=False,
                            path_prefix="results/figures/03b_sampled_chain_page", title="Последовательность порождаемых графов")
plt.show()

**(б) Лучшая запрунённая модель.** Цепочка лучших (после прунинга) моделей; показаны только шаги, на которых модель менялась.

In [ ]:
traj = chain_trajectories(res.chain, ds)
viz.fig_chain_metrics(res.chain, traj); plt.show()
pages = viz.fig_chain_pages(res.chain, traj, ds, per_page=6, ncols=3, dedupe=True,
                            path_prefix="results/figures/03_chain_best_page", title="Лучшая (запрунённая) модель по поколениям")
plt.show()

## 8. Выборка графов и фронт Парето
Каждая точка – уникальная структура из архива; видно компромисс точности и сложности и роль устойчивости.

In [ ]:
viz.fig_pareto(res, true_sc); plt.show()

## 9. Выбор оптимального графа и аппроксимация выборки

Разные деревья могут задавать **одну и ту же функцию** (например, $\alpha x-\beta xy$ и $-\beta x\,(y-\alpha/\beta)$).
Поэтому архив сначала группируется по **классам эквивалентности** (раскрытие скобок и сравнение существенных мономов), и
оптимальный граф выбирается среди лучших представителей классов **по целевой функции $J$**. Top-10 дополнительно
уточняются подгонкой по траектории ОДУ – это проверка, а не критерий выбора (иначе выигрывают переобученные модели
с лишними членами: RSS ниже, но при 21 точке это не значимо).

In [ ]:
reps, counts = class_representatives(res, ds)
n_lv = sum(1 for sc in res.archive.values() if sc.acc < 0.35 and matches_lv(sc.model))
print(f"Синтаксически разных графов в выборке: {len(res.archive)}; классов эквивалентности: {len(reps)}; "
      f"графов в классе Лотки–Вольтерры: {n_lv}")
rows = refine_top(res, ds, topk=10, reps=reps)
rows_J = sorted(rows, key=lambda r: r["scored"].J)
print(f"{'#':>2} {'J':>7} {'J_final':>8} {'RSS':>8}  ЛВ?   раскрытая формула")
for i, r in enumerate(rows_J):
    print(f"{i+1:>2} {r['scored'].J:7.3f} {r['J_final']:8.3f} {r['rss']:8.1f}  {str(r['lv_equiv']):5} {r['expanded']}")
final = rows_J[0]
best_model, info = final["info"]["model"], final["info"]
print("\nВыбранный граф:", best_model)
print("Раскрыто:", expanded_str(best_model))
print("Коэффициенты (α, β, δ, γ):", lv_coefficients(best_model))

In [ ]:
viz.fig_fit(info, best_model, ds); plt.show()

## 10. Разложение целевой функции и чувствительность выбора к весам

In [ ]:
viz.fig_objective_terms(res, 12, arch=reps); plt.show()
viz.fig_weight_sensitivity(res, true_sc, arch=reps); plt.show()

## 11. Интервенции

**(а) Структурная интервенция «взаимная любовь»** (из ЛР № 2): $\mathrm{do}(u=1)$ с 1910 г. добавляет в оба уравнения
$\mathrm{lam}\cdot u\cdot x\cdot y$. Если найденный граф отражает причинную структуру, его предсказание при интервенции
совпадёт с эталоном; конкурирующие графы, одинаково хорошо описывающие *наблюдательные* данные, расходятся.

In [ ]:
models = {"найденный граф": (info["model"], info["x0"], info["y0"], dict(c="C3", lw=2.2)),
          "эталон ЛВ (ЛР №1)": (true_info["model"], true_info["x0"], true_info["y0"], dict(c="k", lw=1.2, ls="--"))}
comp = [sc for sc in reps if not matches_lv(sc.model) and sc.acc < 0.35][:3]
for k, sc in enumerate(comp):
    mi = trajectory_info(sc.model, ds)
    if np.isfinite(mi["rss"]):
        lab = expanded_str(sc.model, 2).replace("dx/dt", "x'").replace("dy/dt", "y'").replace("*", "")
        models[f"конкурент {k+1} (RSS={mi['rss']:.0f}): {lab}"] = (mi["model"], mi["x0"], mi["y0"], dict(c=f"C{k}", lw=1.3))
viz.fig_do_love(models, ds, beta=0.0248); plt.show()

**(б) Интервенции в данные** (как в ЛР № 2): для выбранного графа повторена процедура мини-батчей (60 батчей по 12 из 21 точки),
сравнивается распределение параметров до/после интервенции: $W_1/\sigma$ и статистика $D$ теста Колмогорова–Смирнова
(звёздочка – значимо при поправке Бонферрони). «Шум метода» – та же статистика между двумя независимыми повторами base.
Устойчивая модель слабо реагирует на шум и малые батчи и *ломается* (большие $W_1/\sigma$) на зануление и перемешивание.

In [ ]:
mm, canon_labels = canonical_lv(best_model)   # параметры в раскрытой форме α, β, δ, γ – как в ЛР № 2
fit_model(mm, prob)
scen_tab = {"zero_lynx": zero_lynx(ds), "noise": noise(ds, 1), "noise_cond": noise_cond(ds, 1),
            "shuffle30": shuffle(ds, 1), "small_sample(k=5)": (ds, 5), "bias_+0.5": noise(ds, 1, bias=0.5)}
tab = intervention_table(mm, ds, scen_tab, n_batches=60, k=12)
labels = canon_labels or [f"θ{i+1}" for i in range(tab["n_params"])]
viz.fig_intervention_heatmaps(tab, labels); plt.show()
viz.fig_param_hists(tab, {k: tab["samples"][k] for k in ["noise", "shuffle30", "bias_+0.5"]}, labels); plt.show()

## 12. Граф выбранной модели и матрица инцидентности $A$

In [ ]:
viz.fig_incidence(best_model); plt.show()

In [ ]:
from IPython.display import Image, display
if os.path.exists("results_seeds/seeds.png"):
    display(Image("results_seeds/seeds.png"))   # создаётся run_seeds.py

## 13. Выводы
Запуск: seed 0, 120 особей, 60 поколений; оценено 7428 моделей, в выборке 4422 разных графа (3883 класса эквивалентности).

1. **Структура найдена.** Генетический алгоритм восстанавливает структуру Лотки–Вольтерры уже к 8-му поколению. Выбранный граф после раскрытия скобок: dx/dt = −0.0249·xy + 0.4823·x, dy/dt = 0.0275·xy − 0.9236·y; RSS по траектории 594.8 (ЛР № 2: 595). Дерево записано в факторизованном виде, например (y − 19.4)/(−40.2/x), но это та же функция.
2. **Устойчивость к начальному seed.** В 5 из 5 запусков (80 особей, 40 поколений) выбран класс Лотки–Вольтерры; первое попадание в класс — на поколениях 9–33.
3. **Прунинг.** Парный t-критерий убирает лишние члены, не значимо улучшающие точность, и сохраняет коэффициенты. Лучшая запрунённая модель попадает в популяцию и служит родителем для кроссовера со случайно сгенерированным деревом.
4. **Целевая функция.** J = ln(acc) + 0.03·C + S, где S = ln(1 + Δ/acc). Относительный рост ошибки при интервенциях без логарифма «льстил» неточным моделям: побеждала dx/dt = 20.6 − y. После исправления класс Лотки–Вольтерры оптимален в самой большой области весов (около 25 % площади на рис. 07). При большом весе устойчивости побеждает неточная модель, при малом весе сложности — модели с лишними членами.
5. **Выбор по J, а не по RSS.** Подгонка по траектории отдаёт предпочтение модели с двумя лишними членами (RSS 450 против 595), но при 21 точке это переобучение; по J разрыв узкий (−1.279 против −1.374), поэтому вывод не категоричен.
6. **Интервенция do(u=1).** Найденный граф совпадает с эталоном при всех lam, а конкуренты с близким качеством на наблюдательных данных (в т. ч. с RSS 450) расходятся: интервенция различает модели, которые по данным неразличимы.
7. **Интервенции в данные.** Шум почти не сдвигает параметры (W1/σ ≈ 0.2, на уровне шума метода 0.13–0.21); сильнее всего влияют малые батчи (1.4–2.4), затем zero_lynx и shuffle (0.4–1.8). Реакция слабее, чем в ЛР № 2, потому что здесь подгонка идёт интегральным согласованием, а не по траектории.

**Ограничения.** 21 точка и два периода колебаний; константы интегральной подгонки смещены (α ≈ 0.61 против 0.48 по траектории) и уточняются по траектории; устойчивость оценена на 8 фиксированных возмущённых наборах, числа зависят от этих розыгрышей; веса J подобраны так, чтобы эталон был оптимальным в широкой области.

## 14. Описание вычислительного эксперимента и источники
* Данные: Hudson Bay, заяц–рысь, 1900–1920 (21 точка), те же, что в ЛР № 1–2.
* Методы: генетическое программирование / символьная регрессия (Koza, 1992); интегральное (слабое) согласование
  для идентификации ОДУ (Schaeffer, 2017; Messenger & Bortz, 2021 – Weak SINDy); подгонка по траектории ОДУ (метод trust-region reflective);
  парный $t$-критерий для прунинга; расстояния между распределениями: Колмогоров–Смирнов, Вассерштейн; BIC (Schwarz, 1978)
  как ориентир для веса сложности; $\mathrm{do}$-исчисление (Pearl, 2009) для структурной интервенции.
* Реализация: пакет `lab3/` (numpy, scipy, sympy, matplotlib), собственная реализация ГА.